**Main Cryptojacking Detection Notebook**

Classify and explain the results of the cryptojacking detection model.

In [ ]:
# Batch test-set configuration
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "graph_aware_classifier":
    PROJECT_ROOT = PROJECT_ROOT.parent

TEST_DATA_DIR = PROJECT_ROOT / "ml" / "feature_datasets" / "2gram" / "test"
GRAPH_DATASET_DIR = PROJECT_ROOT / "ml" / "dataset"
OCCURRENCES_DIR = PROJECT_ROOT / "graph_aware_classifier" / "occurences"
BATCH_OUTPUT_DIR = PROJECT_ROOT / "graph_aware_classifier" / "batch_detection_output"

NGRAM_SIZE = 2
SHAP_TOP_FEATURES = 10
TUNED_THRESHOLDS = {
    2: 0.51,
    3: 0.4571270751932118,
    4: 0.532279806931644,
}

if NGRAM_SIZE not in TUNED_THRESHOLDS:
    raise ValueError(f"No tuned classification threshold for {NGRAM_SIZE}-gram model")

CLASS_NAMES = {
    0: "Benign",
    1: "Cryptojacking",
}

In [ ]:
#############################################
# Run the complete RF -> SHAP -> LLM batch
#############################################

import importlib
import json
import re
import sys

import numpy as np
import shap
from scipy.sparse import load_npz

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import graph_aware_classifier.sample_explainer as sample_explainer
import graph_aware_classifier.llm_analyzer as llm_analyzer
from graph_aware_classifier.extract_sample_graph_ngrams import (
    extract_sample_graph_ngrams,
)

sample_explainer = importlib.reload(sample_explainer)
llm_analyzer = importlib.reload(llm_analyzer)

rf, vocabulary = sample_explainer._load_model_and_vocabulary(NGRAM_SIZE)
if len(vocabulary) != rf.n_features_in_:
    raise ValueError(
        f"Vocabulary has {len(vocabulary)} features but the RF expects "
        f"{rf.n_features_in_}"
    )

positive_class_indices = np.flatnonzero(rf.classes_ == 1)
if positive_class_indices.size != 1:
    raise ValueError(f"Expected exactly one class-1 label, found {rf.classes_}")
positive_class_index = int(positive_class_indices[0])
threshold = TUNED_THRESHOLDS[NGRAM_SIZE]
feature_names = [
    " ".join(vocabulary.get_entry(feature_id).ngram)
    for feature_id in range(len(vocabulary))
]
shap_explainer = shap.TreeExplainer(rf)

BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OCCURRENCES_DIR.mkdir(parents=True, exist_ok=True)


def resolve_source_json(row_metadata):
    """Resolve a row's graph JSON from metadata, then repository-relative paths."""
    source_name = row_metadata["source_name"]
    metadata_path = row_metadata.get("source_path")
    if metadata_path:
        candidate = Path(metadata_path)
        if candidate.is_file():
            return candidate.resolve()

    candidate = GRAPH_DATASET_DIR / row_metadata["class_name"] / source_name
    if not candidate.is_file():
        raise FileNotFoundError(
            f"Source graph JSON not found for {source_name!r}; tried "
            f"metadata path {metadata_path!r} and {candidate}"
        )
    return candidate.resolve()


def load_spotted_function_disassembly(source_json_path, shap_evidence):
    """Load full assembly lines for functions referenced by top SHAP n-grams."""
    function_names = {
        function["function"]
        for feature in shap_evidence
        for function in feature["functions"]
        if function["function"] != "UNKNOWN"
    }
    disassembly = {}
    with source_json_path.open("r", encoding="utf-8") as source_file:
        source_graph = json.load(source_file)

    for graph in source_graph.get("graphs", []):
        function_name = graph.get("function")
        if function_name not in function_names:
            continue
        disassembly[function_name] = [
            " ".join(
                part for part in (
                    str(instruction.get("addr", "")),
                    str(instruction.get("mnemonic", "")),
                    str(instruction.get("operands", "")),
                ) if part
            )
            for instruction in graph.get("instructions", [])
        ]

    missing_functions = function_names - disassembly.keys()
    if missing_functions:
        raise ValueError(
            f"SHAP occurrence functions missing from {source_json_path}: "
            f"{sorted(missing_functions)}"
        )
    return disassembly


batch_files = sorted(TEST_DATA_DIR.glob("X_batch_*.npz"))
if not batch_files:
    raise FileNotFoundError(f"No X_batch_*.npz test matrices found in {TEST_DATA_DIR}")

analyzer = None
batch_reports = []
written_report_paths = set()

for matrix_path in batch_files:
    batch_number = int(matrix_path.stem.rsplit("_", 1)[1])
    labels_path = TEST_DATA_DIR / f"y_batch_{batch_number:03d}.npy"
    metadata_path = TEST_DATA_DIR / f"row_metadata_batch_{batch_number:03d}.json"
    if not labels_path.is_file():
        raise FileNotFoundError(f"Labels are missing for {matrix_path.name}: {labels_path}")
    if not metadata_path.is_file():
        raise FileNotFoundError(
            f"Row metadata is missing for {matrix_path.name}: {metadata_path}"
        )

    matrix = load_npz(matrix_path).tocsr()
    labels = np.load(labels_path, allow_pickle=False)
    with metadata_path.open("r", encoding="utf-8") as metadata_file:
        row_metadata = json.load(metadata_file)

    if matrix.shape[1] != rf.n_features_in_:
        raise ValueError(
            f"{matrix_path.name} has {matrix.shape[1]} features; "
            f"the model expects {rf.n_features_in_}"
        )
    if matrix.shape[0] != len(labels) or matrix.shape[0] != len(row_metadata):
        raise ValueError(
            f"Batch {batch_number:03d} row mismatch: matrix={matrix.shape[0]}, "
            f"labels={len(labels)}, metadata={len(row_metadata)}"
        )

    print(f"Processing batch {batch_number:03d}: {matrix.shape[0]} samples")

    for row_index in range(matrix.shape[0]):
        metadata = row_metadata[row_index]
        actual_label = int(labels[row_index])
        metadata_label = int(metadata["label"])
        if metadata_label != actual_label:
            raise ValueError(
                f"Label mismatch in batch {batch_number:03d}, row {row_index}: "
                f"y={actual_label}, metadata={metadata_label}"
            )
        expected_class_name = {
            0: "benign",
            1: "cryptojacking",
        }.get(actual_label)
        if expected_class_name is None:
            raise ValueError(f"Unexpected test label {actual_label} in {metadata_path}")
        if metadata.get("class_name") != expected_class_name:
            raise ValueError(
                f"Class mismatch in batch {batch_number:03d}, row {row_index}: "
                f"label={actual_label}, metadata class={metadata.get('class_name')!r}"
            )
        if int(metadata.get("batch_number", batch_number)) != batch_number:
            raise ValueError(
                f"Batch number mismatch in {metadata_path}, row {row_index}"
            )
        if int(metadata.get("row_index_in_batch", row_index)) != row_index:
            raise ValueError(
                f"Row index mismatch in {metadata_path}, row {row_index}"
            )

        source_path = resolve_source_json(metadata)
        source_name = Path(metadata["source_name"]).name
        binary_name = (
            source_name[:-5]
            if source_name.lower().endswith(".json")
            else source_name
        )
        safe_binary_name = re.sub(r"[^\w.-]", "_", binary_name)
        report_path = BATCH_OUTPUT_DIR / f"{safe_binary_name}_analysis.json"
        if report_path in written_report_paths:
            raise ValueError(
                f"More than one test row maps to report filename {report_path.name}"
            )
        written_report_paths.add(report_path)

        sparse_row = matrix.getrow(row_index)
        malware_probability = float(
            rf.predict_proba(sparse_row)[0, positive_class_index]
        )
        predicted_class = int(malware_probability >= threshold)

        dense_row = sparse_row.toarray().astype(np.float32, copy=False)
        raw_explanation = shap_explainer(dense_row)
        explanation = shap.Explanation(
            values=raw_explanation.values,
            base_values=raw_explanation.base_values,
            data=dense_row,
            feature_names=feature_names,
        )
        top_features = sample_explainer.ranked_feature_contributions(
            explanation,
            class_index=positive_class_index,
            limit=SHAP_TOP_FEATURES,
        )

        occurrence_report = extract_sample_graph_ngrams(
            source_path,
            n=NGRAM_SIZE,
        )
        occurrence_path = OCCURRENCES_DIR / f"{safe_binary_name}_vocabulary.json"
        with occurrence_path.open("w", encoding="utf-8") as occurrence_file:
            json.dump(occurrence_report, occurrence_file, indent=2, ensure_ascii=False)

        shap_evidence = llm_analyzer.build_evidence_payload(
            top_features,
            occurrence_path,
        )
        if predicted_class == 1:
            function_disassembly = load_spotted_function_disassembly(
                source_path,
                shap_evidence,
            )
            if analyzer is None:
                analyzer = llm_analyzer.LLMAnalyzer()
            llm_result = analyzer.analyze_sample(
                binary_name=binary_name,
                rf_prediction=predicted_class,
                rf_probability=malware_probability,
                top_features=top_features,
                occurrences_path=occurrence_path,
                function_disassembly=function_disassembly,
            )
            llm_report = {
                "status": llm_result["status"],
                "failure_reason": llm_result.get("failure_reason"),
                "token_usage": llm_result["token_usage"],
                "result": llm_result,
            }
        else:
            llm_report = {
                "status": "skipped",
                "reason": "The RF prediction was benign at the tuned threshold.",
                "token_usage": {
                    "input_tokens": 0,
                    "output_tokens": 0,
                    "source": "not_called",
                },
            }

        report = {
            "sample": {
                "binary_name": binary_name,
                "source_name": source_name,
                "source_path": source_path.relative_to(PROJECT_ROOT).as_posix(),
                "split": metadata.get("split", "test"),
                "batch_number": batch_number,
                "row_index_in_batch": row_index,
                "actual_label": actual_label,
                "actual_class": expected_class_name,
            },
            "random_forest": {
                "ngram_size": NGRAM_SIZE,
                "threshold": threshold,
                "cryptojacking_probability": malware_probability,
                "predicted_class": predicted_class,
                "predicted_class_name": CLASS_NAMES[predicted_class],
            },
            "shap": {
                "top_feature_limit": SHAP_TOP_FEATURES,
                "top_features": shap_evidence,
            },
            "llm": llm_report,
        }
        with report_path.open("w", encoding="utf-8") as report_file:
            json.dump(report, report_file, indent=2, ensure_ascii=False, allow_nan=False)

        batch_reports.append(report)
        print(
            f"  [{len(batch_reports)}] {binary_name}: "
            f"{CLASS_NAMES[predicted_class]} "
            f"(p={malware_probability:.4f}), LLM {llm_report['status']} -> "
            f"{report_path.name}"
        )

print(
    f"Completed {len(batch_reports)} reports in {BATCH_OUTPUT_DIR}. "
    f"LLM analyzed: {sum(r['llm']['status'] == 'completed' for r in batch_reports)}; "
    f"failed: {sum(r['llm']['status'] == 'failed' for r in batch_reports)}; "
    f"skipped: {sum(r['llm']['status'] == 'skipped' for r in batch_reports)}."
)

Processing batch 000: 100 samples
  [1] x86__64__lsb__unix-system-v__clang-3.8.0__O3__no-obf__unstripped__coreutils-8.30__expand: Benign (p=0.0050), LLM skipped -> x86__64__lsb__unix-system-v__clang-3.8.0__O3__no-obf__unstripped__coreutils-8.30__expand_analysis.json
  [2] mobsync.exe: Benign (p=0.0000), LLM skipped -> mobsync.exe_analysis.json


2026-10-04 18:54:58,039 [INFO] graph_aware_classifier.llm_analyzer — Starting LLM analysis for 00bba2efa9e9bb020828cde686e51ea3a39de47dabf65f197b18e957257f69c0.exe …
2026-10-04 19:30:28,320 [INFO] httpx — HTTP Request: POST http://localhost:11434/api/generate "HTTP/1.1 200 OK"
2026-10-04 19:30:28,777 [INFO] graph_aware_classifier.llm_analyzer — LLM result saved → D:\BSCS\Cryptojacking Detection Thesis\Graph Aware Cryptojacking Classifier\graph_aware_classifier\llm_results\00bba2efa9e9bb020828cde686e51ea3a39de47dabf65f197b18e957257f69c0.exe_llm_result.json


  [3] 00bba2efa9e9bb020828cde686e51ea3a39de47dabf65f197b18e957257f69c0.exe: Cryptojacking (p=0.8700), LLM completed -> 00bba2efa9e9bb020828cde686e51ea3a39de47dabf65f197b18e957257f69c0.exe_analysis.json
  [4] comrepl.exe: Benign (p=0.0050), LLM skipped -> comrepl.exe_analysis.json
  [5] ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O0__no-obf__unstripped__coreutils-8__copy_1_.30__hostid: Benign (p=0.0000), LLM skipped -> ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O0__no-obf__unstripped__coreutils-8__copy_1_.30__hostid_analysis.json
  [6] SlideToShutDown.exe: Benign (p=0.0000), LLM skipped -> SlideToShutDown.exe_analysis.json
  [7] setx.exe: Benign (p=0.0000), LLM skipped -> setx.exe_analysis.json
  [8] x86__64__lsb__unix-system-v__clang-3.8.0__O2__no-obf__unstripped__coreutils-8.30__split: Benign (p=0.0150), LLM skipped -> x86__64__lsb__unix-system-v__clang-3.8.0__O2__no-obf__unstripped__coreutils-8.30__split_analysis.json
  [9] sdiageng.dll: Benign (p=0.0

2026-10-04 19:30:35,064 [INFO] graph_aware_classifier.llm_analyzer — Starting LLM analysis for 6032d1538d68df2b8a118c6c5f3756a9c383d213f71fae50fef2c59482cdf107.exe …


  [10] ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O0__no-obf__unstripped__coreutils-8__copy_1_.30__readlink: Benign (p=0.0000), LLM skipped -> ARM__32__lsb__unix-system-v__arm-linux-gnueabihf-gcc-5.4.0__O0__no-obf__unstripped__coreutils-8__copy_1_.30__readlink_analysis.json


In [ ]:
#############################################
# Research G-Eval: Explanation Evidence Faithfulness
#############################################

from graph_aware_classifier.batch_evaluation import evaluate_batch_reports

research_summary = evaluate_batch_reports(
    batch_reports=batch_reports,
    batch_output_dir=BATCH_OUTPUT_DIR,
    generator=analyzer,
)
print(json.dumps(research_summary, indent=2))

In [ ]:
#############################################
# Batch RF Results Summary
#############################################

import pandas as pd
from IPython.display import display

summary_table = pd.DataFrame([
    {
        "Binary": report["sample"]["binary_name"],
        "Actual": report["sample"]["actual_class"],
        "RF Prediction": report["random_forest"]["predicted_class_name"],
        "Cryptojacking Probability": report["random_forest"]["cryptojacking_probability"],
        "Threshold": report["random_forest"]["threshold"],
        "LLM Status": report["llm"]["status"],
        "G-Eval Status": report.get("evaluation", {}).get("status", "not_run"),
    }
    for report in batch_reports
])
display(summary_table)

In [ ]:
#############################################
# SHAP Contributions Preview
#############################################

if batch_reports:
    shap_preview = pd.DataFrame(batch_reports[0]["shap"]["top_features"])[
        ["ngram", "frequency", "shap_value"]
    ].rename(columns={
        "ngram": "N-Gram Sequence",
        "frequency": "Frequency",
        "shap_value": "SHAP Contribution",
    })
    print(f"SHAP evidence for {batch_reports[0]['sample']['binary_name']}")
    display(shap_preview)
else:
    print("No batch reports were generated.")

In [ ]:
#############################################
# Function and Address Evidence Preview
#############################################

if batch_reports:
    sample_report = batch_reports[0]
    for feature in sample_report["shap"]["top_features"]:
        print(f"N-Gram: {feature['ngram']}")
        print(f"SHAP contribution: {feature['shap_value']:+.6f}")
        print(f"Functions and address groups: {feature['functions']}")
        print("---------------------------------------------")

## Stage 5 — Batch LLM Analysis and G-Eval

The notebook runs explanation generation only for samples classified as cryptojacking by the tuned RF threshold, then evaluates valid explanations against the recorded SHAP evidence and selected function disassembly. Invalid or truncated responses are marked as failures, never benign predictions. Each test sample's RF, SHAP, graph-derived function/address evidence, generation output, and G-Eval result is written to a separate JSON report.

In [ ]:
#############################################
# Batch LLM Analysis Summary
#############################################

llm_reports = [
    report for report in batch_reports
    if report["llm"]["status"] == "completed"
]
print(f"LLM analyses completed: {len(llm_reports)}")
print(
    "LLM analyses skipped: "
    f"{sum(report['llm']['status'] == 'skipped' for report in batch_reports)}"
)
if not llm_reports:
    print("No sample was flagged for LLM analysis by the RF threshold.")

In [ ]:
#############################################
# Example LLM Analysis Result
#############################################

if llm_reports:
    llm_result = llm_reports[0]["llm"]["result"]
    analysis = llm_result.get("analysis", {})
    print(f"LLM result for {llm_reports[0]['sample']['binary_name']}")
    print(f"Tier: {analysis.get('tier', 'N/A')}")
    print(f"Confidence score: {analysis.get('confidence_score', 0.0):.2f}")
    print(f"Backend/model: {llm_result.get('llm_backend', 'N/A')} / {llm_result.get('llm_model', 'N/A')}")
    print("Explanation:")
    print(analysis.get("explanation", "No explanation provided."))
    print("Key indicators:")
    for indicator in analysis.get("key_indicators", []):
        print(f"- {indicator}")
    print(f"Recommended action: {analysis.get('recommended_action', 'N/A')}")
else:
    print("No LLM result is available to display.")

In [ ]:
#############################################
# Example LLM Per-Feature Evidence Analysis
#############################################

if llm_reports:
    evidence_analysis = llm_reports[0]["llm"]["result"].get("analysis", {}).get(
        "evidence_analysis", []
    )
    if evidence_analysis:
        evidence_table = pd.DataFrame(evidence_analysis)
        display_columns = [
            column for column in ["ngram", "shap_contribution", "assessment"]
            if column in evidence_table.columns
        ]
        display(evidence_table[display_columns])
    else:
        print("The LLM response contains no per-feature evidence analysis.")
else:
    print("No LLM result is available to display.")

In [ ]:
#############################################
# Total LLM Tokens Used in This Batch Run
#############################################

completed_llm_reports = [
    report for report in batch_reports
    if report["llm"]["status"] == "completed"
]
token_usage_records = [
    report["llm"].get("token_usage", {})
    for report in completed_llm_reports
]

known_input_counts = [
    usage["input_tokens"]
    for usage in token_usage_records
    if isinstance(usage.get("input_tokens"), int)
]
known_output_counts = [
    usage["output_tokens"]
    for usage in token_usage_records
    if isinstance(usage.get("output_tokens"), int)
]

print(f"LLM requests completed: {len(completed_llm_reports)}")
if known_input_counts:
    print(f"Total input tokens: {sum(known_input_counts):,}")
else:
    print("Total input tokens: unavailable from the configured backend")
if known_output_counts:
    print(f"Total output tokens: {sum(known_output_counts):,}")
else:
    print("Total output tokens: unavailable from the configured backend")
unknown_usage_count = sum(
    usage.get("input_tokens") is None or usage.get("output_tokens") is None
    for usage in token_usage_records
)
if unknown_usage_count:
    print(
        f"Token usage was not reported for {unknown_usage_count} request(s); "
        "totals include only requests with reported counts."
    )